In [ ]:

import json
from collections import defaultdict, Counter
from tqdm import tqdm
from pathlib import Path
from typing import Set, Dict, List, Any, Tuple
import re

# --- 配置区域 ---
FILE_PATHS = {
    "MRCONSO": Path("MRCONSO.RRF"),
    "MRSTY": Path("MRSTY.RRF"),
    "MRREL": Path("MRREL.RRF"),
    "MRDEF": Path("MRDEF.RRF"),
}
OUTPUT_FILE = Path("medical_knowledge_dict_final.json")

TARGET_SEMANTIC_TYPES = {
    "Disease or Syndrome", "Neoplastic Process", "Mental or Behavioral Dysfunction",
    "Congenital Abnormality", "Injury or Poisoning", "Pathologic Function",
    "Sign or Symptom", "Finding", 
    "Diagnostic Procedure", "Therapeutic or Preventive Procedure", 
    "Laboratory Procedure", "Health Care Activity",
    "Pharmacologic Substance", "Medical Device",
    "Body Part, Organ, or Organ Component",
    "Virus", "Bacterium"
}

CHEST_DISEASE_MAPPING = {
    "C0018800": {"name": "Cardiomegaly", "category": "cardiac", "severity": "moderate"},
    "C0032285": {"name": "Pneumonia", "category": "infectious", "severity": "high"},
    "C0013604": {"name": "Edema", "category": "fluid_disorder", "severity": "moderate"},
    "C1334969": {"name": "Lung Opacity", "category": "radiological", "severity": "variable"},
    "C0032227": {"name": "Pleural Effusion", "category": "pleural", "severity": "moderate"},
    "C0016658": {"name": "Fracture", "category": "traumatic", "severity": "high"},
    "C0241721": {"name": "Enlarged Cardiomediastinum", "category": "cardiac", "severity": "moderate"},
    "C0032326": {"name": "Pneumothorax", "category": "pleural", "severity": "high"}
}

TERM_TYPE_PRIORITIES = {
    'PT': 1,    # Preferred Term - 最高优先级
    'SY': 2,    # Synonym
    'AB': 2,    # Abbreviation  
    'FN': 3,    # Full Name
    'BD': 3,    # Brand Name
    'CD': 4,    # Clinical Drug
    'RT': 5,    # Related Term
}

OPTIMIZED_RELATIONSHIP_MAP = {
    'has_symptom': {'key': 'symptoms', 'direction': 'forward'},
    'disease_has_associated_finding': {'key': 'symptoms', 'direction': 'forward'},
    'has_finding': {'key': 'symptoms', 'direction': 'forward'},
    'manifestation_of': {'key': 'symptoms', 'direction': 'inverse'},
    'finding_of': {'key': 'symptoms', 'direction': 'inverse'},
    
    'may_be_treated_by': {'key': 'treatments', 'direction': 'forward'},
    'treated_by': {'key': 'treatments', 'direction': 'forward'},
    'may_treat': {'key': 'treatments', 'direction': 'inverse'},
    
    'may_be_diagnosed_by': {'key': 'diagnostic_methods', 'direction': 'forward'},
    'diagnosed_by': {'key': 'diagnostic_methods', 'direction': 'forward'},
    
    'causes': {'key': 'causes', 'direction': 'forward'},
    'caused_by': {'key': 'caused_by', 'direction': 'forward'},
    'complicates': {'key': 'complications', 'direction': 'forward'},
    'complicated_by': {'key': 'complications', 'direction': 'inverse'},
    
    'has_risk_factor': {'key': 'risk_factors', 'direction': 'forward'},
    'risk_factor_of': {'key': 'risk_factors', 'direction': 'inverse'},
    
    'associated_with': {'key': 'associated_diseases', 'direction': 'forward'},
    'co_occurs_with': {'key': 'comorbidities', 'direction': 'forward'},
    
    'has_location': {'key': 'anatomical_location', 'direction': 'forward'},
    'location_of': {'key': 'anatomical_location', 'direction': 'inverse'},
    
    'isa': {'key': 'parent_concepts', 'direction': 'forward'},
    'inverse_isa': {'key': 'child_concepts', 'direction': 'forward'},
}

def parse_rrf_line(line: str) -> list:
    """解析RRF格式的行"""
    return line.strip().split('|')

def validate_configuration():
    """验证配置和文件"""
    errors = []
    
    for name, path in FILE_PATHS.items():
        if not path.exists():
            errors.append(f"Missing file: {path}")
    
    try:
        OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        errors.append(f"Cannot create output directory: {e}")
    
    if errors:
        print("Configuration errors:")
        for error in errors:
            print(f"  - {error}")
        return False
    
    return True

def get_cui_whitelist_and_types() -> Tuple[Set[str], Dict[str, Set[str]]]:
    valid_cui_set = set()
    cui_to_types_map = defaultdict(set)
    
    with open(FILE_PATHS["MRSTY"], 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc="Pass 1/5: Filtering CUIs by Semantic Type"):
            parts = parse_rrf_line(line)
            if len(parts) < 4: 
                continue
            cui, sty = parts[0], parts[3]
            if sty in TARGET_SEMANTIC_TYPES:
                valid_cui_set.add(cui)
                cui_to_types_map[cui].add(sty)
    
    print(f"Found {len(valid_cui_set)} CUIs matching target semantic types.")
    return valid_cui_set, cui_to_types_map

def build_comprehensive_name_mapping(valid_cui_set: Set[str]) -> Dict[str, str]:
    cui_name_candidates = defaultdict(list)
    
    with open(FILE_PATHS["MRCONSO"], 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc="Building comprehensive name mapping"):
            parts = parse_rrf_line(line)
            if len(parts) < 15: 
                continue
            
            cui, lat, ts, tty, sab, string = (
                parts[0], parts[1], parts[2], parts[12], parts[11], parts[14]
            )
            
            if cui not in valid_cui_set or lat != "ENG":
                continue
            
            if sab in {'ICD9CM', 'ICD10CM', 'ICD9', 'ICD10'}:
                continue
            priority = TERM_TYPE_PRIORITIES.get(tty, 99)
            
            cui_name_candidates[cui].append({
                "name": string,
                "type": tty,
                "priority": priority,
                "is_preferred": (tty == 'PT' and ts == 'P'),
                "source": sab
            })
    cui_to_name = {}
    for cui, candidates in cui_name_candidates.items():
        preferred_terms = [c for c in candidates if c["is_preferred"]]
        if preferred_terms:
            cui_to_name[cui] = preferred_terms[0]["name"]
        else:
            candidates.sort(key=lambda x: (x["priority"], len(x["name"])))
            cui_to_name[cui] = candidates[0]["name"] if candidates else f"CUI_{cui}"
    
    print(f"Built name mapping for {len(cui_to_name)} CUIs")
    return cui_to_name

def process_terms_and_codes(valid_cui_set: Set[str], comprehensive_name_map: Dict[str, str]) -> Dict[str, Any]:
    knowledge_dict = defaultdict(lambda: {
        "terms": [],
        "icd_codes": {
            "icd_9": [],
            "icd_10": []
        }
    })
    
    with open(FILE_PATHS["MRCONSO"], 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc="Pass 2/5: Processing Terms & Codes"):
            parts = parse_rrf_line(line)
            if len(parts) < 15: 
                continue
            
            cui, lat, ts, tty, code, sab, string = (
                parts[0], parts[1], parts[2], parts[12], 
                parts[13], parts[11], parts[14]
            )
            
            if cui not in valid_cui_set or lat != "ENG":
                continue
            if code and sab in {'ICD9CM', 'ICD10CM', 'ICD9', 'ICD10'}:
                if sab in {'ICD9CM', 'ICD9'}:
                    icd_version = "icd_9"
                    priority = 1 if sab == 'ICD9CM' else 2
                else:  
                    icd_version = "icd_10"
                    priority = 1 if sab == 'ICD10CM' else 2
                code_info = {
                    "code": code,
                    "description": string,
                    "source": sab,
                    "priority": priority
                }
                knowledge_dict[cui]["icd_codes"][icd_version].append(code_info)
            else:
                priority = TERM_TYPE_PRIORITIES.get(tty, 99)
                term_info = {
                    "term": string,
                    "type": tty,
                    "priority": priority,
                    "source": sab,
                    "is_preferred": (tty == 'PT' and ts == 'P')
                }
                
                knowledge_dict[cui]["terms"].append(term_info)
    
    for cui, data in knowledge_dict.items():
        for icd_version in ["icd_9", "icd_10"]:
            if data["icd_codes"][icd_version]:
                codes = data["icd_codes"][icd_version]
                unique_codes = {}
                for code_info in codes:
                    code = code_info["code"]
                    if code not in unique_codes or code_info["priority"] < unique_codes[code]["priority"]:
                        unique_codes[code] = code_info
                data["icd_codes"][icd_version] = sorted(
                    unique_codes.values(), 
                    key=lambda x: (x["priority"], x["code"])
                )
        
        if data["terms"]:
            unique_terms = {}
            for term_info in data["terms"]:
                term_key = term_info["term"].lower()
                if term_key not in unique_terms or term_info["priority"] < unique_terms[term_key]["priority"]:
                    unique_terms[term_key] = term_info
            
            data["terms"] = sorted(unique_terms.values(), key=lambda x: x["priority"])
    
    print(f"Processed terms and codes for {len(knowledge_dict)} CUIs.")
    return knowledge_dict

def add_definitions(knowledge_dict: Dict[str, Any], valid_cui_set: Set[str]):
    """添加定义，优先级处理"""
    definition_sources_priority = {
        'MSH': 1, 'NCI': 2, 'SNOMEDCT_US': 3, 'FMA': 4, 'HPO': 5
    }
    with open(FILE_PATHS["MRDEF"], 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc="Pass 3/5: Adding Definitions"):
            parts = parse_rrf_line(line)
            if len(parts) < 6: 
                continue
            cui, source, definition = parts[0], parts[4], parts[5] 
            if cui in valid_cui_set and definition and len(definition.strip()) > 10:
                if "definitions" not in knowledge_dict[cui]:
                    knowledge_dict[cui]["definitions"] = []    
                priority = definition_sources_priority.get(source, 99)
                knowledge_dict[cui]["definitions"].append({
                    "definition": definition.strip(),
                    "source": source,
                    "priority": priority
                })
    for cui, data in knowledge_dict.items():
        if "definitions" in data and data["definitions"]:
            # 按优先级和长度排序
            data["definitions"].sort(key=lambda x: (x["priority"], -len(x["definition"])))
            data["best_definition"] = data["definitions"][0]["definition"]
            del data["definitions"]  # 删除中间数据
    
    print("Finished adding definitions.")

def build_relationship_network(knowledge_dict: Dict[str, Any], valid_cui_set: Set[str], 
                             comprehensive_name_map: Dict[str, str]):
    relationship_counter = Counter()
    added_relations = defaultdict(set)
    with open(FILE_PATHS["MRREL"], 'r', encoding='utf-8', errors='ignore') as f:
        for line in tqdm(f, desc="Pass 4/5: Building Relationship Network"):
            parts = parse_rrf_line(line)
            if len(parts) < 11: 
                continue
            cui1, rela, cui2 = parts[0], parts[7], parts[4]
            if (cui1 not in valid_cui_set or cui2 not in valid_cui_set or 
                not rela or rela in ['', 'null']):
                continue
            relation_config = OPTIMIZED_RELATIONSHIP_MAP.get(rela)
            if relation_config:
                key = relation_config['key']
                source_cui, target_cui = (
                    (cui1, cui2) if relation_config['direction'] == 'forward' 
                    else (cui2, cui1)
                )
            else:
                key = rela  
                source_cui, target_cui = cui1, cui2
            relation_tuple = (source_cui, key, target_cui)
            if relation_tuple in added_relations[source_cui]:
                continue
            
            if source_cui in knowledge_dict:
                if "relationships" not in knowledge_dict[source_cui]:
                    knowledge_dict[source_cui]["relationships"] = defaultdict(list)
                target_name = comprehensive_name_map.get(target_cui, f"Concept_{target_cui}")
                
                relationship_info = {
                    "cui": target_cui,
                    "name": target_name,
                    "relation_type": rela
                }
                knowledge_dict[source_cui]["relationships"][key].append(relationship_info)
                added_relations[source_cui].add(relation_tuple)
                relationship_counter[key] += 1
    
    print("Relationship distribution:")
    for rel_type, count in relationship_counter.most_common(15):
        print(f"  - {rel_type}: {count}")

def calculate_importance_score(cui: str, data: dict, cui_types: Set[str]) -> float:
    score = 0.0
    # 语义类型权重
    semantic_weights = {
        "Disease or Syndrome": 1.0,
        "Neoplastic Process": 0.95,
        "Sign or Symptom": 0.85,
        "Finding": 0.8,
        "Diagnostic Procedure": 0.75,
        "Therapeutic or Preventive Procedure": 0.8,
        "Pharmacologic Substance": 0.7
    }
    max_semantic_weight = max(
        (semantic_weights.get(stype, 0.4) for stype in cui_types), 
        default=0.4
    )
    score += max_semantic_weight
    icd_score = 0
    for version in ["icd_9", "icd_10"]:
        codes = data["icd_codes"].get(version, [])
        for code in codes:
            if code.get("source") == "ICD10CM":
                icd_score += 0.5
            elif code.get("source") in ["ICD10", "ICD9CM"]:
                icd_score += 0.3
            else:
                icd_score += 0.1
    score += min(icd_score, 1.0)
    if "relationships" in data:
        total_relations = sum(len(rels) for rels in data["relationships"].values())
        score += min(total_relations * 0.05, 0.8)
    unique_terms = len(set(t["term"].lower() for t in data.get("terms", [])))
    score += min(unique_terms * 0.02, 0.3)
    if cui in CHEST_DISEASE_MAPPING:
        score += 0.5
    
    return round(min(score, 5.0), 3)

def finalize_and_save(knowledge_dict: Dict[str, Any], cui_to_types: Dict[str, Set[str]]):
    """最终处理并保存"""
    final_dict = {}
    stats = {
        "total_concepts": 0,
        "with_icd9": 0,
        "with_icd10": 0,
        "with_any_icd": 0,
        "core_chest_diseases": 0,
        "semantic_distribution": Counter(),
        "relation_distribution": Counter()
    }
    for cui, data in tqdm(knowledge_dict.items(), desc="Pass 5/5: Finalizing Dictionary"):
        # 必须有术语才处理
        if not data.get("terms"):
            continue
        preferred_terms = [t for t in data["terms"] if t.get("is_preferred")]
        if preferred_terms:
            preferred_name = preferred_terms[0]["term"]
        else:
            preferred_name = data["terms"][0]["term"]  
        final_record = {
            "preferred_name": preferred_name,
            "semantic_types": sorted(list(cui_to_types[cui])),
            "importance_score": calculate_importance_score(cui, data, cui_to_types[cui])
        }
        if "best_definition" in data:
            final_record["definition"] = data["best_definition"]
        terms_by_type = {
            "preferred": preferred_name,
            "synonyms": [],
            "abbreviations": [],
            "alternatives": []
        }
        
        for term in data["terms"]:
            if term.get("is_preferred"):
                continue  # 已经设置为preferred
            elif term["type"] == "SY":
                terms_by_type["synonyms"].append(term["term"])
            elif term["type"] == "AB":
                terms_by_type["abbreviations"].append(term["term"])
            else:
                terms_by_type["alternatives"].append(term["term"])
        for key in ["synonyms", "abbreviations", "alternatives"]:
            unique_terms = list(dict.fromkeys(terms_by_type[key]))  
            terms_by_type[key] = unique_terms[:5]  
        
        final_record["terms"] = terms_by_type
        if any(data["icd_codes"].values()):
            final_record["icd_codes"] = {}
            for version in ["icd_9", "icd_10"]:
                if data["icd_codes"][version]:
                    final_record["icd_codes"][version] = data["icd_codes"][version]
                    if version == "icd_9":
                        stats["with_icd9"] += 1
                    else:
                        stats["with_icd10"] += 1
            stats["with_any_icd"] += 1
        if "relationships" in data:
            filtered_relations = {}
            for rel_type, relations in data["relationships"].items():
                # 限制每种关系类型最多5个
                top_relations = relations[:5]
                if top_relations:
                    filtered_relations[rel_type] = top_relations
                    stats["relation_distribution"][rel_type] += len(top_relations)
            
            if filtered_relations:
                final_record["relationships"] = filtered_relations
        if cui in CHEST_DISEASE_MAPPING:
            chest_info = CHEST_DISEASE_MAPPING[cui]
            final_record["chest_disease_info"] = {
                "is_core_chest_disease": True,
                "disease_category": chest_info["category"],
                "clinical_severity": chest_info["severity"]
            }
            stats["core_chest_diseases"] += 1
        for stype in final_record["semantic_types"]:
            stats["semantic_distribution"][stype] += 1
        
        final_dict[cui] = final_record
        stats["total_concepts"] += 1
        
    print("\n=== Final Dictionary Statistics ===")
    print(f"Total concepts: {stats['total_concepts']}")
    print(f"With ICD-9: {stats['with_icd9']}")
    print(f"With ICD-10: {stats['with_icd10']}")
    print(f"With any ICD: {stats['with_any_icd']}")
    print(f"Core chest diseases: {stats['core_chest_diseases']}")
    
    print("\nTop semantic types:")
    for stype, count in stats["semantic_distribution"].most_common(10):
        print(f"  - {stype}: {count}")
    
    print("\nTop relationship types:")
    for rel_type, count in stats["relation_distribution"].most_common(10):
        print(f"  - {rel_type}: {count}")
    
    # 保存文件
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(final_dict, f, indent=2, ensure_ascii=False)
    
    print(f"\nDictionary saved to: {OUTPUT_FILE}")
    print(f"File size: {OUTPUT_FILE.stat().st_size / (1024*1024):.1f} MB")

def main():
    print("Building Medical Knowledge Dictionary for ICD Auto-coding")
    print("=" * 60)
    
    if not validate_configuration():
        print("Please fix configuration errors before proceeding.")
        return
    
    try:
        valid_cuis, cui_types = get_cui_whitelist_and_types()
        comprehensive_name_map = build_comprehensive_name_mapping(valid_cuis)
        knowledge_base = process_terms_and_codes(valid_cuis, comprehensive_name_map)
        add_definitions(knowledge_base, valid_cuis)
        build_relationship_network(knowledge_base, valid_cuis, comprehensive_name_map)
        finalize_and_save(knowledge_base, cui_types)
        
        print("\n🎉 Medical knowledge dictionary construction completed successfully!")
        
    except Exception as e:
        print(f"\n❌ Error during processing: {e}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main()
